# 11. 보드와 이야기하기 — 시리얼(pyserial)

USB 로 꽂은 보드(ESP32 등)와 **글자 한 줄씩** 주고받는다. 약속은 "한 줄 = 명령 하나" 뿐이고, 명령 이름은 펌웨어가 정한다.

**먼저**
- edge-lab 의 [보드] 앱이 포트를 쓰고 있으면 여기서 못 연다 — 보드 앱에서 연결을 끊거나 edge-lab 을 끈다.
- 보드는 USB 로만 연결한다. 서보·모터처럼 전류가 큰 부품의 전원은 USB 에서 끌어오지 말고 따로 준다.
- ESP32 의 핀은 3.3V 다. 5V 센서를 바로 잇지 않는다.

## 1. 포트 찾기

In [ ]:
import os
import time
import serial
import serial.tools.list_ports

ports = list(serial.tools.list_ports.comports())
for p in ports:
    print(f"{p.device:14s} {p.description}  (VID {p.vid:04X})" if p.vid else f"{p.device:14s} {p.description}")
PORT = os.environ.get("BOARD_PORT") or (ports[0].device if ports else None)
print("쓸 포트:", PORT)

## 2. 열기
`BAUD` 는 펌웨어와 같아야 한다 (보통 115200). DTR/RTS 를 내려 두면 여는 순간 보드가 리셋되는 것을 막는다(보드에 따라 다름).

In [ ]:
BAUD = 115200
EOL = "\n"                                   # 줄 끝 — 펌웨어가 \r\n 을 기다리면 바꾼다

ser = serial.Serial()
ser.port, ser.baudrate, ser.timeout = PORT, BAUD, 0.2
ser.dtr = ser.rts = False
ser.open()
time.sleep(0.3)
ser.reset_input_buffer()
print("열림:", ser.name)

## 3. 보내기 · 보내고 답 받기
보드가 명령을 그대로 되돌려 보내는(echo) 경우가 있다 — 보낸 글과 같은 줄은 답으로 치지 않는다.

In [ ]:
def send(text):
    ser.write((text + EOL).encode("utf-8"))

def ask(text, wait=1.0):
    """보내고, 그 뒤 처음 들어온 줄을 돌려준다. 없으면 None."""
    ser.reset_input_buffer()
    send(text)
    end = time.time() + wait
    while time.time() < end:
        line = ser.readline().decode("utf-8", errors="replace").strip()
        if line and line != text:
            return line
    return None

print("답:", ask("PING"))

## 4. 보드가 스스로 보내는 줄 읽기 (센서 값 등) — 5초

In [ ]:
end = time.time() + 5
while time.time() < end:
    line = ser.readline().decode("utf-8", errors="replace").strip()
    if line:
        print(f"{time.strftime('%H:%M:%S')}  {line}")

## 5. 닫기 — 다른 프로그램(보드 앱)이 쓰려면 꼭 닫는다

In [ ]:
ser.close()
print("닫힘")

## 해 볼 것
1. 펌웨어가 센서 값을 `T=23.5` 처럼 보내면 숫자만 꺼내 리스트에 모으고 `matplotlib` 으로 그려 보자.
2. 06 번의 손동작 결과를 보드로 보내 LED 를 켜 보자 (예: 엄지척 → `LED ON`).
3. `ask` 에 머리말(prefix) 인자를 더해 `OK` 로 시작하는 줄만 답으로 받게 바꿔 보자.